# Notebook : Analyse de la Five-Point Task

Ce notebook est dédié à l'exploration des données du paradigme **Five-Point**.

Étant donné qu'il s'agit d'une tâche relativement nouvelle avec une littérature limitée, nous sommes en terrain inconnu ! Cela rend l'analyse passionnante car le champ d'exploration est vaste, mais cela peut aussi être délicat car nous ne savons pas exactement *a priori* ce que nous allons trouver.

**Notre approche :**
Nous vous fournirons les fondations nécessaires pour démarrer :
* Les codes d'événements (triggers).
* Des propositions de clusters de canaux.
* Des fenêtres temporelles d'intérêt potentielles.

C'est à vous de prendre le relais et d'explorer vos propres hypothèses. Pour la structure du code (chargement, nettoyage, extraction), vous pouvez vous inspirer ou réutiliser les fonctions des notebooks précédents (`02_analysis1_gonogo.ipynb` ou `03_analysis1_textsemantic.ipynb`).

---

### Rappel : Qu'est-ce que le paradigme Five-Point ?

Le **Five-Point Test** est une tâche de **créativité motrice** et de **fluence non-verbale**. Les participants doivent créer autant de dessins uniques que possible en reliant 5 points disposés de manière fixe (comme la face '5' d'un dé) avec un stylet.

**Déroulement d'un essai :**
1.  **Présentation du stimulus** (les 5 points) → Trigger `Onset`
2.  **Phase d'idéation** (planification mentale du dessin)
3.  **Premier contact** (début de l'exécution motrice) → Trigger `FirstStroke`
4.  **Exécution** (tracé du dessin)
5.  **Complétion** (fin du dessin)

---

### Pistes d'exploration : Composantes cérébrales d'intérêt

Nous vous fournirons le code de base pour les ERPs classiques (N2 et P3). Cependant, la nature motrice et cognitive de la tâche suggère que d'autres composantes pourraient être très pertinentes.

> **Note :** L'analyse des composantes ci-dessous peut nécessiter un prétraitement ou un epoching spécifique (ex: aligner les epochs sur la réponse motrice `FirstStroke` plutôt que sur le stimulus `Onset`, ou utiliser des fenêtres de temps plus longues).

#### 1. MRCP (Movement-Related Cortical Potential)
L'activité préparatoire au mouvement.
- **Définition** : Potentiel cortical lent précédant et accompagnant le mouvement volontaire.
- **Latence** : Commence environ 1 à 2 secondes **AVANT** le mouvement.
- **Topographie** : Maximum sur les électrodes centrales (**Cz, C3, C4**) — cortex moteur et prémoteur.
- **Phases clés** :
  - **Bereitschaftspotential (BP)** : Potentiel de préparation précoce (~1.5s à 0.5s avant).
  - **NS' (Negative Slope)** : Rampe négative tardive (~0.5s à 0s avant).
  - **Motor Potential (MP)** : Pic négatif juste avant/pendant l'exécution.
- **Interprétation** : Reflète la planification et l'activation progressive du cortex moteur.



#### 2. CNV (Contingent Negative Variation)
L'attente et l'anticipation.
- **Définition** : Onde lente négative qui se développe entre un stimulus d'avertissement et un stimulus impératif (l'action).
- **Latence** : Se développe entre le stimulus (`Onset`) et l'action anticipée.
- **Topographie** : Maximum fronto-central (**Fz, F3, F4, Cz**).
- **Composantes** :
  - **CNV précoce** : Orientation de l'attention (~300-600 ms après le stimulus).
  - **CNV tardive** : Préparation motrice (~600 ms jusqu'à l'action).
- **Interprétation** : Anticipation, préparation motrice et motivation.

#### 3. ERD/ERS (Event-Related Desynchronization/Synchronization)
La dynamique oscillatoire (Analyse Temps-Fréquence).
- **Définition** : Changements de puissance dans les oscillations cérébrales liés à l'événement.
- **Bandes de fréquence clés** :
  - **Mu (8-13 Hz)** : Rythme sensorimoteur (au repos).
  - **Bêta (13-30 Hz)** : Rythme moteur actif.
- **Patterns typiques** :
  - **ERD (Désynchronisation)** : **Diminution** de puissance (Mu/Bêta) pendant la préparation et l'exécution du dessin.
  - **ERS (Synchronisation)** : **Augmentation** de puissance (rebond Bêta) juste après la fin du mouvement.
- **Interprétation** :
  - ERD = Activation du cortex sensorimoteur.
  - ERS = Inhibition post-mouvement / Réinitialisation du système moteur.

---

### Objectif du Notebook

Il n'est pas obligatoire d'atteindre l'étape du Machine Learning pour réussir ce module. **Identifier des ERPs clairs** ou des dynamiques oscillatoires (ERD/ERS) possédant une **interprétation neuroscientifique pertinente** constitue déjà un excellent résultat pour une tâche exploratoire comme celle-ci.

In [2]:
FIVEPOINT_EVENT_CODES = {
    "START": 14,
    "onset":       12,  # 5-dot matrix appears
    "first_touch": 11,  # first pen contact = movement onset
    "retouch":     16,  # additional strokes within the same design
    "END":         13,  # trial/design window ended
}

In [3]:
CHANNEL_CLUSTERS = {
    # Fronto-central midline cluster
    # ------------------------------
    # Fz + Cz together approximate FCz / midline SMA–ACC–M1 sources.
    # We expect here:
    #   - N2 (conflict / "don't repeat that design")
    #   - ERN / error-related negativities
    #   - CNV / Bereitschaftspotential (pre-movement slow negativity)
    #   - frontal-midline theta (4–7 Hz) linked to monitoring & control
    "midline_fc": ["Fz", "Cz"],

    # Parietal posterior cluster
    # --------------------------
    # P3 + P4 capture bilateral parietal cortex.
    # We expect here:
    #   - P3b / late positive complex for "design decided / checked"
    #   - parietal alpha (8–12 Hz) reflecting visual attention vs internal focus
    "parietal": ["P3", "P4"],

    # Left motor cortex (contralateral to right hand drawing)
    # -------------------------------------------------------
    # C3 ≈ right hand area of M1.
    # We expect here:
    #   - late phase of the RP (lateralized readiness potential)
    #   - mu (8–13 Hz) ERD during drawing
    #   - beta (15–25 Hz) ERD + contralateral PMBR
    "motor_left": ["C3"],

    # Right motor cortex (ipsilateral to right hand drawing)
    # ------------------------------------------------------
    # C4 is used as:
    #   - ipsilateral motor comparison site for lateralization (C3 vs C4)
    #   - control site for bilateral beta ERD/ERS patterns
    "motor_right": ["C4"],
}

In [4]:
ERP_MEAN_WINDOWS = [
    {
        "name": "N2_mean_200_350",
        "cluster": "midline_fc",
        "tmin": 0.200,
        "tmax": 0.350,
    },
    {
        "name": "P3_mean_300_600_midline_fc",
        "cluster": "midline_fc",
        "tmin": 0.300,
        "tmax": 0.600,
    },
    {
        "name": "P3_mean_300_600_parietal",
        "cluster": "parietal",
        "tmin": 0.300,
        "tmax": 0.600,
    },
]

ERP_PEAK_WINDOWS = [
    {
        "name": "N2_peak_200_350",
        "cluster": "midline_fc",
        "tmin": 0.200,
        "tmax": 0.350,
        "mode": "min",  # negative peak (N2 / ERN-like)
    },
    {
        "name": "P3_peak_300_600",
        "cluster": "parietal",
        "tmin": 0.300,
        "tmax": 0.600,
        "mode": "max",  # positive peak (P3 / Pe-like)
    },
]